# N-gram, Bag of Words, TF-IDF, Embedding, and Transformer: Simple Guide

This notebook compares five common text-representation methods using simple words and practical code.

We compare them on:
- Similarity quality (lexical vs semantic)
- Complexity (runtime and memory)
- Vocabulary handling (OOV, rare words, context)

You will also get:
- Interview tips for each method
- Enterprise app tips for production usage
- A final "When to Use Which" summary

In [ ]:
import time
import tracemalloc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.decomposition import TruncatedSVD

# Shared tiny corpus for consistent comparisons
corpus = [
    "machine learning improves search relevance",
    "deep learning helps image recognition",
    "natural language processing powers chatbots",
    "search engines use ranking algorithms",
    "support tickets need smart routing",
    "fraud detection needs anomaly signals",
    "chatbots handle customer service questions",
    "vector databases enable semantic retrieval",
    "transformers understand context in text",
    "tf idf is a strong baseline for retrieval",
]

# Sentence pairs to evaluate similarity quality
pairs = [
    ("chatbot handles support questions", "customer service chatbot answers tickets"),
    ("image model classifies photos", "database indexing for text search"),
    ("semantic retrieval with vectors", "vector database for similarity search"),
    ("fraud alert from unusual behavior", "anomaly detection for financial fraud"),
]

train_docs, test_docs = train_test_split(corpus, test_size=0.3, random_state=42)


def sparse_memory_bytes(x):
    """Approximate memory in bytes for scipy sparse or numpy dense arrays."""
    if hasattr(x, "data") and hasattr(x, "indices") and hasattr(x, "indptr"):
        return x.data.nbytes + x.indices.nbytes + x.indptr.nbytes
    return x.nbytes


def timed_run(fn, *args, **kwargs):
    """Run a function and return (result, elapsed_seconds, peak_memory_bytes)."""
    tracemalloc.start()
    t0 = time.perf_counter()
    result = fn(*args, **kwargs)
    elapsed = time.perf_counter() - t0
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return result, elapsed, peak


def pairwise_scores(embed_fn, sent_pairs):
    scores = []
    for a, b in sent_pairs:
        va, vb = embed_fn([a, b])
        score = float(cosine_similarity(va.reshape(1, -1), vb.reshape(1, -1))[0, 0])
        scores.append(score)
    return scores


print("Train docs:", len(train_docs), "| Test docs:", len(test_docs))

## 2) N-gram: Token Sequences, Similarity, Complexity, and Vocabulary

### Simple idea
N-gram means we look at word chunks that appear together.
- Unigram: one word (`support`)
- Bigram: two words (`support ticket`)
- Trigram: three words (`support ticket routing`)

### Similarity
- Better than plain single words when phrase order matters a bit.
- Still mostly lexical similarity (same/similar words), not deep meaning.

### Complexity
- As `n` grows, vocabulary size grows fast.
- Vectors become very large and sparse.

### Vocabulary handling
- New unseen phrases become out-of-vocabulary quickly.
- Rare n-grams are common, especially for trigrams.

### Interview tips
- Say: "N-grams add local word-order signal to BoW features."
- Mention trade-off: higher `n` can improve phrase capture but can overfit.
- Common follow-up: "How to control feature explosion?" Answer with `min_df`, `max_features`, and pruning.

### Enterprise tips
- Useful for query understanding, phrase matching, and intent hints.
- Works well in search logs and routing rules when data is limited.
- Use preprocessing to normalize casing, punctuation, and variants for stability.

In [ ]:
# N-gram demo
ngram_vec = CountVectorizer(ngram_range=(1, 2), min_df=1)
X_ng = ngram_vec.fit_transform(corpus)

print("N-gram matrix shape:", X_ng.shape)
print("Approx memory (KB):", round(sparse_memory_bytes(X_ng) / 1024, 2))

sample_pair = [pairs[0][0], pairs[0][1]]
P_ng = ngram_vec.transform(sample_pair)
ng_score = float(cosine_similarity(P_ng[0], P_ng[1])[0, 0])
print("Sample pair cosine similarity (n-gram):", round(ng_score, 4))

for n in [(1, 1), (1, 2), (1, 3)]:
    v = CountVectorizer(ngram_range=n)
    Xm = v.fit_transform(corpus)
    print(f"ngram_range={n} -> vocab_size={len(v.vocabulary_)}, shape={Xm.shape}")

## 3) Bag of Words: Count Vectors, Similarity, Complexity, and Vocabulary

### Simple idea
Bag of Words (BoW) just counts words. It ignores order.

### Similarity
- Good for lexical overlap.
- If two sentences use different words for same meaning, score may be low.

### Complexity
- Fast to train and transform.
- Still sparse and can be large if vocabulary is large.

### Vocabulary handling
- OOV words are ignored at inference.
- Rare words can create noisy features unless filtered.

### Interview tips
- Say: "BoW is a strong baseline because it is simple, fast, and interpretable."
- Mention sparse matrices and preprocessing importance.
- Follow-up: "Why no semantics?" Because BoW does not model synonym meaning or context.

### Enterprise tips
- Great first baseline for classification and routing.
- Works in moderation, tagging, and support triage pipelines.
- Easy to explain to non-ML stakeholders.

In [ ]:
# Bag of Words demo
bow_vec = CountVectorizer()
X_bow = bow_vec.fit_transform(corpus)

print("BoW matrix shape:", X_bow.shape)
print("BoW sparsity (% zeros):", round(100 * (1 - X_bow.nnz / (X_bow.shape[0] * X_bow.shape[1])), 2))

P_bow = bow_vec.transform(sample_pair)
bow_score = float(cosine_similarity(P_bow[0], P_bow[1])[0, 0])
print("Sample pair cosine similarity (BoW):", round(bow_score, 4))

print("Top 15 vocabulary terms:", list(bow_vec.vocabulary_.keys())[:15])

## 4) TF-IDF: Weighted Term Vectors, Similarity, Complexity, and Vocabulary

### Simple idea
TF-IDF keeps word counts but down-weights common words and up-weights informative words.

Formula:

$$
\mathrm{tfidf}(t,d) = \mathrm{tf}(t,d) \cdot \log\left(\frac{N}{df(t)}\right)
$$

### Similarity
- Usually better than raw counts for retrieval and ranking.
- Still mostly lexical, but weighting gives cleaner similarity.

### Complexity
- Similar sparse complexity as BoW plus IDF statistics.
- Easy to scale in classic search/retrieval pipelines.

### Vocabulary handling
- OOV terms are ignored during transform.
- Stopword handling and token normalization matter a lot.

### Interview tips
- Say: "TF-IDF often beats BoW when common words should matter less."
- Mention `sublinear_tf=True` and stopword choices.
- Follow-up: "Can it capture synonyms?" Mostly no, unless words overlap.

### Enterprise tips
- Strong for search ranking baselines, ticket dedup, FAQ retrieval.
- Cheap, explainable, and fast to deploy in existing systems.
- Good bootstrap stage before expensive semantic models.

In [ ]:
# TF-IDF demo
tfidf_vec = TfidfVectorizer(sublinear_tf=True)
X_tfidf = tfidf_vec.fit_transform(corpus)

print("TF-IDF matrix shape:", X_tfidf.shape)
print("Approx memory (KB):", round(sparse_memory_bytes(X_tfidf) / 1024, 2))

P_tfidf = tfidf_vec.transform(sample_pair)
tfidf_score = float(cosine_similarity(P_tfidf[0], P_tfidf[1])[0, 0])
print("Sample pair cosine similarity (TF-IDF):", round(tfidf_score, 4))
print("Difference vs BoW:", round(tfidf_score - bow_score, 4))

## 5) Embedding: Dense Vectors, Similarity, Complexity, and Vocabulary

### Simple idea
Embeddings map text into dense vectors (small numeric lists) where similar meanings are closer.

### Similarity
- Better semantic similarity than BoW/TF-IDF in many cases.
- Can connect related words even with less exact overlap.

### Complexity
- More compute than BoW/TF-IDF, but still manageable.
- Dense vectors are compact compared with huge sparse vocab vectors.

### Vocabulary handling
- Static embeddings may fail on unknown words.
- Subword-based models improve OOV handling.

### Interview tips
- Distinguish static embeddings vs contextual embeddings.
- Mention pooling choices (mean/max/CLS for sentence vectors).
- Follow-up: "Why dense vectors?" Better semantic geometry and lower dimension.

### Enterprise tips
- Good for semantic search and duplicate detection.
- Useful in clustering, recommendations, and multilingual matching.
- Store vectors in a vector index for fast nearest-neighbor search.

In [ ]:
# Embedding demo with safe fallback
# First try sentence-transformers. If unavailable, build dense vectors via TF-IDF + SVD.

embedding_backend = ""

try:
    from sentence_transformers import SentenceTransformer
    st_model = SentenceTransformer("all-MiniLM-L6-v2")

    def embed_texts(texts):
        return np.array(st_model.encode(texts, normalize_embeddings=True))

    embedding_backend = "sentence-transformers"
except Exception as e:
    print("sentence-transformers not available, using fallback TF-IDF + SVD.")
    print("Reason:", str(e)[:160])

    fallback_tfidf = TfidfVectorizer()
    X_fb = fallback_tfidf.fit_transform(corpus)
    svd_dim = min(50, X_fb.shape[1] - 1) if X_fb.shape[1] > 1 else 1
    svd = TruncatedSVD(n_components=svd_dim, random_state=42)
    svd.fit(X_fb)

    def embed_texts(texts):
        dense = svd.transform(fallback_tfidf.transform(texts))
        norms = np.linalg.norm(dense, axis=1, keepdims=True) + 1e-12
        return dense / norms

    embedding_backend = "tfidf+svd fallback"

E_pair = embed_texts(sample_pair)
embed_score = float(cosine_similarity(E_pair[0].reshape(1, -1), E_pair[1].reshape(1, -1))[0, 0])

print("Embedding backend:", embedding_backend)
print("Sample pair cosine similarity (Embedding):", round(embed_score, 4))
print("Embedding vector dimension:", E_pair.shape[1])

## 6) Transformer: Contextual Embeddings, Similarity, Complexity, and Vocabulary

### Simple idea
Transformers read all words with attention, so each word gets meaning from context.

### Similarity
- Best semantic quality among methods here in many real tasks.
- Handles polysemy better (same word, different meanings by context).

### Complexity
- Highest compute and memory cost in this list.
- Latency and token limits matter in production.

### Vocabulary handling
- Uses subword tokenization, so unseen words can still be split and understood partly.
- Better robustness for rare/misspelled/compound words than word-only vocab.

### Interview tips
- Explain self-attention simply: each token can "look at" other tokens.
- Mention why context helps: meaning depends on surrounding words.
- Follow-up: trade-off is accuracy vs inference cost/latency.

### Enterprise tips
- Excellent for semantic search, RAG retrieval, intent matching.
- Add caching, batching, and model quantization for lower latency/cost.
- Governance: monitor drift, data privacy, and prompt/content policy controls.

In [ ]:
# Transformer sentence embedding demo (safe optional dependency)
transformer_ready = False

try:
    from sentence_transformers import SentenceTransformer
    transformer_model = SentenceTransformer("all-MiniLM-L6-v2")

    def transformer_embed(texts):
        return np.array(transformer_model.encode(texts, normalize_embeddings=True))

    transformer_ready = True
except Exception as e:
    print("Transformer model not available in this environment.")
    print("Reason:", str(e)[:160])

if transformer_ready:
    T_pair = transformer_embed(sample_pair)
    transformer_score = float(cosine_similarity(T_pair[0].reshape(1, -1), T_pair[1].reshape(1, -1))[0, 0])
    print("Sample pair cosine similarity (Transformer):", round(transformer_score, 4))
    print("Transformer embedding dimension:", T_pair.shape[1])
else:
    transformer_score = np.nan
    print("Skipping transformer scoring. Install sentence-transformers to enable.")

## 7) Side-by-Side Benchmark: Similarity Quality vs Runtime/Memory

This section benchmarks all methods on the same sentence pairs and reports:
- Average cosine similarity on related pairs
- Runtime for feature generation
- Peak memory during run
- Approx feature memory size

In [ ]:
def ngram_embed(texts):
    return ngram_vec.transform(texts).toarray()


def bow_embed(texts):
    return bow_vec.transform(texts).toarray()


def tfidf_embed(texts):
    return tfidf_vec.transform(texts).toarray()


def benchmark_method(name, fit_fn, embed_fn, feature_obj_getter=None):
    _, elapsed, peak = timed_run(fit_fn)
    scores = pairwise_scores(embed_fn, pairs)
    avg_score = float(np.mean(scores))

    feature_mem = np.nan
    if feature_obj_getter is not None:
        try:
            feature_mem = sparse_memory_bytes(feature_obj_getter())
        except Exception:
            feature_mem = np.nan

    return {
        "method": name,
        "avg_similarity": round(avg_score, 4),
        "fit_time_ms": round(elapsed * 1000, 2),
        "peak_memory_kb": round(peak / 1024, 2),
        "feature_memory_kb": round(feature_mem / 1024, 2) if not np.isnan(feature_mem) else np.nan,
    }


results = []

results.append(
    benchmark_method(
        "N-gram",
        lambda: ngram_vec.fit_transform(corpus),
        ngram_embed,
        lambda: X_ng,
    )
)

results.append(
    benchmark_method(
        "Bag of Words",
        lambda: bow_vec.fit_transform(corpus),
        bow_embed,
        lambda: X_bow,
    )
)

results.append(
    benchmark_method(
        "TF-IDF",
        lambda: tfidf_vec.fit_transform(corpus),
        tfidf_embed,
        lambda: X_tfidf,
    )
)

results.append(
    benchmark_method(
        "Embedding",
        lambda: embed_texts(corpus),
        embed_texts,
        lambda: np.asarray(embed_texts(corpus)),
    )
)

if transformer_ready:
    results.append(
        benchmark_method(
            "Transformer",
            lambda: transformer_embed(corpus),
            transformer_embed,
            lambda: np.asarray(transformer_embed(corpus)),
        )
    )
else:
    results.append(
        {
            "method": "Transformer",
            "avg_similarity": np.nan,
            "fit_time_ms": np.nan,
            "peak_memory_kb": np.nan,
            "feature_memory_kb": np.nan,
        }
    )

benchmark_df = pd.DataFrame(results)
benchmark_df

In [ ]:
# Plot quality vs cost
plot_df = benchmark_df.dropna(subset=["avg_similarity", "fit_time_ms"]).copy()

plt.figure(figsize=(8, 5))
plt.scatter(plot_df["fit_time_ms"], plot_df["avg_similarity"], s=120)

for _, r in plot_df.iterrows():
    plt.annotate(r["method"], (r["fit_time_ms"], r["avg_similarity"]), xytext=(6, 4), textcoords="offset points")

plt.xlabel("Runtime (ms)")
plt.ylabel("Average Similarity")
plt.title("Quality vs Cost Across Methods")
plt.grid(alpha=0.3)
plt.show()

## 8) When to Use Which Method (Practical Summary Matrix)

### Compact comparison table

| Method | Similarity Quality | Complexity | Vocabulary Handling | Interpretability | Typical Enterprise Use |
|---|---|---|---|---|---|
| N-gram | Low-Medium (lexical + phrase overlap) | Low-Medium, sparse grows fast with n | Weak for unseen phrases | High | Phrase matching, query analytics, routing rules |
| Bag of Words | Low-Medium (lexical only) | Low, sparse | OOV ignored | Very High | Fast baseline classifiers, moderation, ticket routing |
| TF-IDF | Medium (weighted lexical) | Low-Medium, sparse + IDF stats | OOV ignored, better weighting | High | Search ranking baseline, dedup, FAQ retrieval |
| Embedding | Medium-High (semantic) | Medium | Better (depends on model/subword) | Medium | Semantic search, clustering, recommendations |
| Transformer | High (contextual semantic) | High | Strong with subword tokenization | Medium-Low | RAG retrieval, intent matching, enterprise assistants |

Rule of thumb:
- Start with TF-IDF if you need a cheap and strong baseline.
- Move to Embedding/Transformer when meaning matters more than exact words.
- Keep BoW/N-gram where interpretability and speed are top priority.

In [ ]:
decision_matrix = pd.DataFrame(
    [
        {
            "method": "N-gram",
            "best_for": "short phrase signals and local order",
            "similarity_strength": "lexical phrase overlap",
            "complexity": "low-medium, sparse explosion with larger n",
            "vocabulary_behavior": "many rare/OOV phrases",
            "interview_talking_points": "captures local order, but high-dimensional sparse",
            "enterprise_use_cases": "query rewriting, phrase match, rule-assisted routing",
        },
        {
            "method": "Bag of Words",
            "best_for": "simple baseline models",
            "similarity_strength": "exact word overlap",
            "complexity": "low, sparse vectors",
            "vocabulary_behavior": "ignores unseen terms",
            "interview_talking_points": "fast and interpretable baseline",
            "enterprise_use_cases": "moderation, intent baseline, ticket classification",
        },
        {
            "method": "TF-IDF",
            "best_for": "retrieval/ranking baseline",
            "similarity_strength": "weighted lexical overlap",
            "complexity": "low-medium",
            "vocabulary_behavior": "good weighting, still lexical",
            "interview_talking_points": "down-weights common words, strong cheap baseline",
            "enterprise_use_cases": "search, FAQ matching, dedup support tickets",
        },
        {
            "method": "Embedding",
            "best_for": "semantic similarity with moderate cost",
            "similarity_strength": "semantic closeness",
            "complexity": "medium dense vectors",
            "vocabulary_behavior": "depends on pretrained vocab/subwords",
            "interview_talking_points": "dense vectors, static vs contextual trade-off",
            "enterprise_use_cases": "semantic search, clustering, recommendations",
        },
        {
            "method": "Transformer",
            "best_for": "highest quality semantic understanding",
            "similarity_strength": "contextual semantic",
            "complexity": "high compute + latency",
            "vocabulary_behavior": "strong via subword tokenization",
            "interview_talking_points": "self-attention gives context, expensive inference",
            "enterprise_use_cases": "RAG retrieval, intent routing, assistant backends",
        },
    ]
)

decision_matrix

### Final "When to Use Which" Summary

Use this checklist:
1. Need fast and explainable baseline? Choose **Bag of Words** or **TF-IDF**.
2. Need phrase sensitivity ("credit card" vs "card credit")? Add **N-grams**.
3. Need semantic matching (similar meaning, different words)? Use **Embeddings**.
4. Need best context understanding and can afford cost? Use **Transformers**.
5. Need strict latency/cost controls? Start simple, then upgrade only where gain is clear.

### Mini Interview Rapid-Fire (8 Q/A)

1. **Q:** Why does TF-IDF often beat raw counts?  
   **A:** It reduces weight of common words and highlights informative terms.

2. **Q:** What is the main weakness of BoW?  
   **A:** It ignores word order and deeper meaning.

3. **Q:** When do n-grams help most?  
   **A:** When short phrases carry meaning, like product names or intents.

4. **Q:** Why are embeddings called dense vectors?  
   **A:** They use relatively small continuous vectors instead of huge sparse vectors.

5. **Q:** Static vs contextual embeddings?  
   **A:** Static gives one vector per word; contextual changes by sentence context.

6. **Q:** Why do transformers handle polysemy better?  
   **A:** They use context from nearby words with attention.

7. **Q:** Biggest production concern for transformers?  
   **A:** Inference cost and latency at scale.

8. **Q:** Good enterprise rollout strategy?  
   **A:** Start with TF-IDF baseline, measure, then add embedding/transformer for high-value paths only.